Ten skrypt oczyszcza i transformuje dane w hurtowni

In [0]:
%sql
MERGE INTO staging.car AS car
USING (
  SELECT
    Inventory.inventory_id,
    CASE WHEN Car.last_update > Inventory.last_update THEN Car.last_update ELSE Inventory.last_update END AS new_valid_to
  FROM landing.car AS Car
  INNER JOIN landing.inventory AS Inventory ON Car.car_id = Inventory.car_id
) AS src
ON car.car_inventory_id = src.inventory_id
AND (car.last_update < src.new_valid_to)
WHEN MATCHED THEN
  UPDATE SET
    car.last_update=src.new_valid_to,
    stag_update_date = CURRENT_TIMESTAMP()

In [0]:
%sql
INSERT INTO staging.car (car_inventory_id, car_id, car_producer, car_model, car_production_year, car_fuel_type, car_license_plates, car_current_rental_rate, car_purchase_price, car_sell_price, last_update,stag_insert_date, stag_update_date)
SELECT
    Inventory.inventory_id AS car_inventory_id,
    Car.car_id,
    Car.producer,
    Car.model,
    Inventory.production_year,
    Inventory.fuel_type,
    Inventory.license_plates,
    Car.rental_rate AS current_rental_rate,
    Inventory.purchase_price,
    Inventory.sell_price,
    CASE WHEN Car.last_update > Inventory.last_update THEN Car.last_update
        ELSE Inventory.last_update END AS last_update,
    CURRENT_TIMESTAMP() AS stag_insert_date,
    NULL AS dw_update_date
FROM landing.car Car
INNER JOIN landing.inventory Inventory ON Car.car_id = Inventory.car_id
LEFT JOIN dw.dim_car Dim_car ON Dim_car.car_inventory_id = Inventory.inventory_id AND Dim_car.is_active = true
WHERE Dim_car.car_inventory_id IS NULL;

In [0]:
%sql
UPDATE staging.car
SET car_fuel_type = 
  CASE LEFT(LOWER(car_fuel_type), 1)
    WHEN 'd' THEN 'Diesel'
    WHEN 'p' THEN 'Petrol'
    ELSE NULL END
WHERE car_fuel_type not in ('Diesel', 'Petrol');

In [0]:
%sql
UPDATE staging.car
SET car_purchase_price = 111000
WHERE car_purchase_price = 111;

In [0]:
%sql
  CREATE OR REPLACE TABLE staging.equipment_groups AS
  SELECT ROW_NUMBER() OVER (ORDER BY eq_ids, eq_group_names) AS eq_group_id,
  eq_ids, 
  eq_group_names
  FROM (
  SELECT distinct eq_ids,eq_group_names
  FROM
    (
    SELECT
    inventory_id,
    string_agg(CAST(equipment_id AS STRING), ', ') WITHIN GROUP (ORDER BY equipment_id ASC) AS eq_ids,
    string_agg(CAST(eq_name AS STRING), ', ') WITHIN GROUP (ORDER BY eq_name ASC) AS eq_group_names
    FROM (
      SELECT ie.inventory_id, ie.equipment_id, CONCAT(name, ' ', type, COALESCE(version, '')) AS eq_name
      FROM landing.inventory_equipment ie
      INNER JOIN landing.equipment e ON ie.equipment_id = e.equipment_id
      ) t
    GROUP BY inventory_id
    ) a
  ) d


In [0]:
%sql
CREATE OR REPLACE TABLE staging.bridge_equipment__equipment_groups AS
SELECT 
  teg.eq_group_id,
  teg.eq_ids,
  teg.eq_group_names,
  CAST(EXPLODED.equipment_id AS INT) AS equipment_id
FROM staging.inventory_equipment_groups teg
LATERAL VIEW EXPLODE(SPLIT(teg.eq_ids, ', ')) EXPLODED AS equipment_id;


In [0]:
%sql
CREATE OR REPLACE TABLE staging.inventory_equipment_groups AS
SELECT i.inventory_id, eq_group.eq_group_id
FROM (
    SELECT 
        inventory_id,
        string_agg(CAST(equipment_id AS STRING), ', ') WITHIN GROUP (ORDER BY equipment_id ASC) AS eq_ids,
        string_agg(CAST(eq_name AS STRING), ', ') WITHIN GROUP (ORDER BY eq_name ASC) AS eq_group_names
    FROM (
      SELECT ie.inventory_id, ie.equipment_id, CONCAT(name, ' ', type, COALESCE(version, '')) AS eq_name
      FROM landing.inventory_equipment ie
      INNER JOIN landing.equipment e ON ie.equipment_id = e.equipment_id
      ) t
    GROUP BY inventory_id
) i
LEFT JOIN staging.equipment_groups eq_group ON i.eq_ids = eq_group.eq_ids